# Hybrid Quantum-Classical Rebalancing (Business Notebook)

This notebook is a business-friendly version of the PoC script.

It demonstrates:
- simulated and real market data modes
- quantum-assisted portfolio weight generation
- classical optimization with risk and transaction cost controls
- KPI summary for decision support

In [1]:
from __future__ import annotations

from dataclasses import dataclass
import logging
from typing import Tuple

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

## 1) Configuration
Adjust these parameters to test different scenarios.

In [ ]:
@dataclass
class PoCConfig:
    data_source: str = "simulated" # change to "real" for real data
    real_tickers: Tuple[str, ...] = ("SPY", "TLT", "GLD")
    real_period: str = "2y"
    real_interval: str = "1d"
    n_assets: int = 3
    n_steps: int = 180
    lookback: int = 20
    random_search_samples: int = 80
    risk_aversion: float = 8.0
    transaction_cost: float = 0.002
    seed: int = 7
    log_level: str = "INFO"


def configure_logging(level: str) -> logging.Logger:
    logger = logging.getLogger("hybrid_rebalancing_notebook")
    resolved_level = getattr(logging, level.upper(), logging.INFO)
    logger.setLevel(resolved_level)

    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(
            logging.Formatter(
                "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
                datefmt="%H:%M:%S",
            )
        )
        logger.addHandler(handler)

    logger.propagate = False
    return logger

## 2) Core Functions
These are the same algorithms as in the script version.

In [3]:
def quantum_policy(features: np.ndarray, theta: np.ndarray) -> np.ndarray:
    n_assets = len(features)
    qc = QuantumCircuit(n_assets)

    for qubit, value in enumerate(features):
        qc.ry(float(value), qubit)

    for qubit in range(n_assets - 1):
        qc.cx(qubit, qubit + 1)

    for qubit in range(n_assets):
        qc.ry(float(theta[qubit]), qubit)
        qc.rz(float(theta[n_assets + qubit]), qubit)

    for qubit in range(n_assets - 1, 0, -1):
        qc.cx(qubit, qubit - 1)

    for qubit in range(n_assets):
        qc.ry(float(theta[2 * n_assets + qubit]), qubit)

    probabilities = Statevector.from_instruction(qc).probabilities()
    p_one = np.zeros(n_assets, dtype=float)
    for basis, prob in enumerate(probabilities):
        for asset in range(n_assets):
            if (basis >> asset) & 1:
                p_one[asset] += prob

    raw_scores = p_one + 1e-9
    return raw_scores / raw_scores.sum()


def simulate_market(n_steps: int, n_assets: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    returns = np.zeros((n_steps, n_assets))

    drift_regimes = np.array([
        [0.0012, 0.0008, 0.0010],
        [-0.0003, 0.0014, 0.0006],
        [0.0016, -0.0002, 0.0009],
    ])[:, :n_assets]

    base_cov = np.array([
        [0.00030, 0.00009, 0.00006],
        [0.00009, 0.00024, 0.00008],
        [0.00006, 0.00008, 0.00028],
    ])
    covariance = base_cov[:n_assets, :n_assets]

    regime = 0
    for t in range(n_steps):
        if t > 0 and t % 45 == 0:
            regime = (regime + 1) % drift_regimes.shape[0]
        returns[t] = rng.multivariate_normal(mean=drift_regimes[regime], cov=covariance)

    return returns


def fetch_real_market_returns(
    tickers: Tuple[str, ...],
    period: str,
    interval: str,
) -> np.ndarray:
    import yfinance as yf

    history = yf.download(
        list(tickers),
        period=period,
        interval=interval,
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    if history.empty:
        raise RuntimeError("No data returned from Yahoo Finance for the selected tickers/period")

    if "Close" in history:
        close_prices = history["Close"]
    elif "Adj Close" in history:
        close_prices = history["Adj Close"]
    else:
        raise RuntimeError("Yahoo Finance response does not contain close prices")

    if getattr(close_prices, "ndim", 1) == 1:
        close_prices = close_prices.to_frame(name=tickers[0])

    close_prices = close_prices.dropna(how="any")
    returns_frame = close_prices.pct_change().dropna(how="any")
    if returns_frame.empty:
        raise RuntimeError("Insufficient price history to compute returns")

    for ticker in tickers:
        if ticker not in returns_frame.columns:
            raise RuntimeError(f"Ticker '{ticker}' missing in Yahoo Finance response")

    ordered = returns_frame.loc[:, list(tickers)]
    return ordered.to_numpy(dtype=float)


def build_features(window_returns: np.ndarray) -> np.ndarray:
    mean_signal = window_returns.mean(axis=0)
    volatility = window_returns.std(axis=0) + 1e-9
    momentum = window_returns[-5:].mean(axis=0)
    zscore = (momentum - mean_signal) / volatility
    scaled = np.tanh(3.5 * zscore)
    return np.pi * scaled


def evaluate_candidate(
    theta: np.ndarray,
    features: np.ndarray,
    mu: np.ndarray,
    cov: np.ndarray,
    prev_weights: np.ndarray,
    risk_aversion: float,
    transaction_cost: float,
) -> Tuple[float, np.ndarray]:
    candidate_weights = quantum_policy(features=features, theta=theta)
    expected_return = float(candidate_weights @ mu)
    variance = float(candidate_weights @ cov @ candidate_weights)
    turnover = float(np.abs(candidate_weights - prev_weights).sum())
    utility = expected_return - risk_aversion * variance - transaction_cost * turnover
    return utility, candidate_weights


def optimize_quantum_parameters(
    rng: np.random.Generator,
    features: np.ndarray,
    mu: np.ndarray,
    cov: np.ndarray,
    prev_weights: np.ndarray,
    theta_center: np.ndarray,
    random_search_samples: int,
    risk_aversion: float,
    transaction_cost: float,
) -> Tuple[np.ndarray, np.ndarray, float]:
    best_theta = theta_center.copy()
    best_utility, best_weights = evaluate_candidate(
        theta=best_theta,
        features=features,
        mu=mu,
        cov=cov,
        prev_weights=prev_weights,
        risk_aversion=risk_aversion,
        transaction_cost=transaction_cost,
    )

    for _ in range(random_search_samples):
        candidate_theta = theta_center + rng.normal(0.0, 0.35, size=theta_center.shape[0])
        utility, candidate_weights = evaluate_candidate(
            theta=candidate_theta,
            features=features,
            mu=mu,
            cov=cov,
            prev_weights=prev_weights,
            risk_aversion=risk_aversion,
            transaction_cost=transaction_cost,
        )
        if utility > best_utility:
            best_utility = utility
            best_theta = candidate_theta
            best_weights = candidate_weights

    return best_theta, best_weights, best_utility

## 3) Simulation Runner
Returns both detailed logs and top-level KPIs.

In [4]:
def run_hybrid_rebalancing(config: PoCConfig):
    logger = configure_logging(config.log_level)

    rng = np.random.default_rng(config.seed)
    if config.data_source.lower() == "real":
        returns = fetch_real_market_returns(
            tickers=config.real_tickers,
            period=config.real_period,
            interval=config.real_interval,
        )
        n_assets = returns.shape[1]
        n_steps = returns.shape[0]
        logger.info(
            "Starting run | source=real assets=%d steps=%d lookback=%d tickers=%s samples=%d risk_aversion=%.3f transaction_cost=%.5f seed=%d",
            n_assets,
            n_steps,
            config.lookback,
            ",".join(config.real_tickers),
            config.random_search_samples,
            config.risk_aversion,
            config.transaction_cost,
            config.seed,
        )
    else:
        returns = simulate_market(
            n_steps=config.n_steps,
            n_assets=config.n_assets,
            seed=config.seed,
        )
        n_assets = config.n_assets
        n_steps = config.n_steps
        logger.info(
            "Starting run | source=simulated assets=%d steps=%d lookback=%d samples=%d risk_aversion=%.3f transaction_cost=%.5f seed=%d",
            n_assets,
            n_steps,
            config.lookback,
            config.random_search_samples,
            config.risk_aversion,
            config.transaction_cost,
            config.seed,
        )

    if n_steps <= config.lookback:
        raise ValueError(
            f"Not enough observations ({n_steps}) for lookback window ({config.lookback})"
        )

    theta = rng.normal(0.0, 0.5, size=3 * n_assets)
    weights = np.full(n_assets, 1.0 / n_assets)

    portfolio_returns = []
    wealth = [1.0]
    turnover_log = []
    utility_log = []
    weight_log = []

    for t in range(config.lookback, n_steps):
        window = returns[t - config.lookback : t]
        features = build_features(window)
        mu = window.mean(axis=0)
        cov = np.cov(window, rowvar=False)
        prev_weights = weights.copy()

        theta, new_weights, utility = optimize_quantum_parameters(
            rng=rng,
            features=features,
            mu=mu,
            cov=cov,
            prev_weights=weights,
            theta_center=theta,
            random_search_samples=config.random_search_samples,
            risk_aversion=config.risk_aversion,
            transaction_cost=config.transaction_cost,
        )

        turnover = float(np.abs(new_weights - weights).sum())
        realized = float(new_weights @ returns[t] - config.transaction_cost * turnover)

        weights = new_weights
        portfolio_returns.append(realized)
        wealth.append(wealth[-1] * (1.0 + realized))
        turnover_log.append(turnover)
        utility_log.append(utility)
        weight_log.append(weights.copy())

        logger.info(
            "day=%3d | utility=%+.6f realized=%+.6f turnover=%.4f wealth=%.4f",
            t,
            utility,
            realized,
            turnover,
            wealth[-1],
        )
        logger.debug(
            "day=%3d | features=%s mu=%s diag_cov=%s prev_w=%s new_w=%s",
            t,
            np.array2string(features, precision=4, suppress_small=True),
            np.array2string(mu, precision=6, suppress_small=True),
            np.array2string(np.diag(cov), precision=7, suppress_small=True),
            np.array2string(prev_weights, precision=4, suppress_small=True),
            np.array2string(new_weights, precision=4, suppress_small=True),
        )

    portfolio_returns_arr = np.array(portfolio_returns)
    weight_log_arr = np.array(weight_log)

    ann_factor = 252
    avg_daily = float(portfolio_returns_arr.mean())
    vol_daily = float(portfolio_returns_arr.std() + 1e-12)
    sharpe = float(np.sqrt(ann_factor) * avg_daily / vol_daily)
    cumulative_return = float(wealth[-1] - 1.0)

    summary = {
        "data_source": config.data_source,
        "tickers": config.real_tickers,
        "assets": n_assets,
        "steps": n_steps,
        "lookback": config.lookback,
        "final_wealth": float(wealth[-1]),
        "cumulative_return_pct": 100.0 * cumulative_return,
        "avg_daily_return_pct": 100.0 * avg_daily,
        "daily_volatility_pct": 100.0 * vol_daily,
        "annualized_sharpe": sharpe,
        "avg_turnover": float(np.mean(turnover_log)),
        "avg_utility": float(np.mean(utility_log)),
    }

    return summary, wealth, weight_log_arr

## 4) Run and View KPI Output
Set `data_source` to `"simulated"` or `"real"`, then execute this cell.

In [5]:
config = PoCConfig(
    data_source="real",  # change to "simulated" for synthetic regimes
    real_tickers=("SPY", "TLT", "GLD"),
    real_period="2y",
    real_interval="1d",
    n_assets=3,
    n_steps=180,
    lookback=20,
    random_search_samples=80,
    risk_aversion=8.0,
    transaction_cost=0.002,
    seed=7,
    log_level="INFO",
)
summary, wealth, weight_log = run_hybrid_rebalancing(config)

print("Hybrid quantum-classical dynamic rebalancing PoC")
print("=" * 56)
print(f"Data source: {summary['data_source']}")
if summary["data_source"].lower() == "real":
    print(f"Tickers: {', '.join(summary['tickers'])}")
print(f"Assets: {summary['assets']}, Steps: {summary['steps']}, Lookback: {summary['lookback']}")
print(f"Final wealth: {summary['final_wealth']:.4f}")
print(f"Cumulative return: {summary['cumulative_return_pct']:.2f}%")
print(f"Average daily return: {summary['avg_daily_return_pct']:.3f}%")
print(f"Daily volatility: {summary['daily_volatility_pct']:.3f}%")
print(f"Sharpe (annualized): {summary['annualized_sharpe']:.2f}")
print(f"Average turnover per rebalance: {summary['avg_turnover']:.3f}")
print(f"Average utility score: {summary['avg_utility']:.6f}")

15:26:15 | INFO | hybrid_rebalancing_notebook | Starting run | source=real assets=3 steps=500 lookback=20 tickers=SPY,TLT,GLD samples=80 risk_aversion=8.000 transaction_cost=0.00200 seed=7
15:26:15 | INFO | hybrid_rebalancing_notebook | day= 20 | utility=-0.001326 realized=+0.001895 turnover=0.4399 wealth=1.0019
15:26:15 | INFO | hybrid_rebalancing_notebook | day= 21 | utility=-0.000453 realized=-0.015786 turnover=0.1643 wealth=0.9861
15:26:15 | INFO | hybrid_rebalancing_notebook | day= 22 | utility=-0.001639 realized=+0.004741 turnover=0.1694 wealth=0.9908
15:26:15 | INFO | hybrid_rebalancing_notebook | day= 23 | utility=-0.001394 realized=+0.001914 turnover=0.1211 wealth=0.9927
15:26:15 | INFO | hybrid_rebalancing_notebook | day= 24 | utility=-0.000997 realized=+0.005473 turnover=0.0933 wealth=0.9981
15:26:15 | INFO | hybrid_rebalancing_notebook | day= 25 | utility=-0.000916 realized=+0.008921 turnover=0.0022 wealth=1.0070
15:26:15 | INFO | hybrid_rebalancing_notebook | day= 26 | uti

Hybrid quantum-classical dynamic rebalancing PoC
Data source: real
Tickers: SPY, TLT, GLD
Assets: 3, Steps: 500, Lookback: 20
Final wealth: 1.2212
Cumulative return: 22.12%
Average daily return: 0.045%
Daily volatility: 0.770%
Sharpe (annualized): 0.92
Average turnover per rebalance: 0.126
Average utility score: 0.000613


## 5) Last 5 Rebalances
Shows the final portfolio allocations generated by the model.

In [6]:
n_last = 5
start_idx = max(0, len(weight_log) - n_last)

print("Last 5 rebalances (weights):")
for i, row in enumerate(weight_log[-n_last:], start=start_idx + 1):
    weight_text = ", ".join(f"{w:.3f}" for w in row)
    print(f"t={i:3d} -> [{weight_text}]")

Last 5 rebalances (weights):
t=476 -> [0.312, 0.334, 0.354]
t=477 -> [0.350, 0.424, 0.226]
t=478 -> [0.532, 0.373, 0.095]
t=479 -> [0.537, 0.386, 0.078]
t=480 -> [0.244, 0.603, 0.153]
